In [2]:
# Password-protected file access (to be used across Jupyter notebooks)

import msoffcrypto # To access password-protected files
import pandas as pd # To read and manipulate data
from io import BytesIO # Temp handling of decrypted files
from getpass import getpass # Get password input w/out encoding

password = getpass('Enter anon file password: ') # Prompt for password
decrypted_file = BytesIO() # Creates temp file in memory

with open(r'C:\Users\karin\Documents\2. Data\Anonymous_Data.xlsx', 'rb') as f: # Open the password-protected file
    office_file = msoffcrypto.OfficeFile(f) # Create Officefile object
    office_file.load_key(password=password) # Load password
    office_file.decrypt(decrypted_file) # Decrypt file into memory

df = pd.read_excel(decrypted_file) # Read decrypted file into a DataFrame (DF)
df.shape # Display the shape of the DF (rows and columns)

(2839, 88)

In [3]:
# Rebuild df_binary for attendance feature below

fail_categories = ['Fail Resit', 'Fail Withdraw', 'Repeat without Attendance', 'Repeat with Attendance', 'Complete Repeat'] # Defining categories which count as a fail
df_binary = df[df['Progression Decision'] != 'Trail Progress'].copy() # Create a new DF excluding 'Trail Progress' rows as it is its own edge case
df_binary['initially_failed'] = df_binary['Progression Decision'].isin(fail_categories) # Create new column in new DF: True if student failed and false if they passed

df_binary.shape # Display the shape of the DF (rows and columns)

(2837, 89)

In [4]:
# Attendance Feature: already numeric but accouting for off-site students with null values

df_binary['is_offsite'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary['is_offsite'].value_counts() # Count the number of off-site students (True) and on-site students (False)

is_offsite
False    2785
True       52
Name: count, dtype: int64

In [5]:
# Attendance Feature: check if any off-site students are categorised as on-site via admin student status

df_binary['no_attendance_data'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary[df_binary['no_attendance_data']]['Student Status'].value_counts() # Count no. students off-site but categorised via admin student status

Student Status
Off-site     41
PR/Repeat    10
Normal        1
Name: count, dtype: int64

In [6]:
# Breakdown of student numbers via three student statuses

df_binary['is_offsite'] = df_binary['Student Status'].str.contains('Off-site', case=False, na=False) # Create new column which turns true if student status contains 'Off-site' (off-site student)
df_binary['is_repeating'] = df_binary['Student Status'].str.contains('PR/Repeat', case=False, na=False) # Create new column which turns true if student status contains 'PR/Repeat' (repeating student)
df_binary['unexplained_null_attendance'] = df_binary['no_attendance_data'] & ~df_binary['is_offsite'] & ~df_binary['is_repeating'] # Create new column which turns true if student has no attendance data but is not off-site or repeating

df_binary[['is_offsite', 'is_repeating', 'unexplained_null_attendance']].sum() # Count students in each category

is_offsite                     42
is_repeating                   92
unexplained_null_attendance     1
dtype: int64

In [7]:
# Dividing repeating students into two categories: those with and without attendance expectation

df_binary['is_repeating'] = df_binary['is_repeating'] # All repeating students for general repeat flag
df_binary['repeating_with_no_attendance_expectation'] = df_binary['is_repeating'] & df_binary['Attendance (%)'].isnull() # Those repeating without attendance expectation
df_binary['repeating_with_attendance'] = df_binary['is_repeating'] & df_binary['Attendance (%)'].notnull() # Those repeating with attendance expectation

df_binary[['repeating_with_no_attendance_expectation', 'repeating_with_attendance']].sum() # Count students in each category

repeating_with_no_attendance_expectation    10
repeating_with_attendance                   82
dtype: int64

In [8]:
# Rebuild list of VLE columns

vle_columns = [col for col in df.columns if 'VLE' in col] # List VLE columns
len(vle_columns) # Display the number of VLE columns

19

In [9]:
# VLE Engagement Score

rag_map = {'RED': 0, 'AMBER': 1, 'GREEN': 2} # Ordinal mapping for RAG values, the higher, the better the engagement score

vle_numeric = df_binary[vle_columns].apply(lambda col: col.str.upper()) # Convert all RAG values to uppercase to avoid errors
vle_numeric = vle_numeric.replace(rag_map).replace('GREY', pd.NA) # Replace RAG values with numeric values and replace GREY with null
vle_numeric = vle_numeric.apply(pd.to_numeric, errors='coerce') # Convert all values to numeric

df_binary['vle_avg_score'] = vle_numeric.mean(axis=1) # Calculate the average VLE score for each student across all weeksand store in a new column
df_binary['vle_red_weeks'] = (vle_numeric == 0).sum(axis=1) # Count the number of RED weeks for each student
df_binary['vle_grey_weeks'] = vle_numeric.isnull().sum(axis=1) # Count the number of GREY weeks for each student
df_binary['has_grey_vle'] = df_binary['vle_grey_weeks'] > 3 # Binary flag check if student has more than 3 GREY weeks (adjusted as fewer likely reflects late registration)

df_binary[['vle_avg_score', 'vle_red_weeks', 'vle_grey_weeks', 'has_grey_vle']].describe() # Print the scores

,vle_avg_score,vle_red_weeks,vle_grey_weeks
count,2746.000000,2837.000000,2837.000000
mean,1.315146,3.348255,1.381036
std,0.499003,4.320137,4.126316
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,1.421053,2.000000,0.000000
75%,1.736842,5.000000,0.000000
max,2.000000,19.000000,19.000000


In [10]:
# Flag for no VLE data students

df_binary['no_vle_data'] = df_binary['vle_avg_score'].isnull() # Pulls 91 students who's overall VLE engagement is null (grey or non-existant)
df_binary['no_vle_data'].sum() # Check for 91 returned

91

In [11]:
# Encoding categoricals

categorical_cols = ['Department', 'Student Status', 'Year Group'] # Columns that need to be one-hot encoded - Added Year Group
df_encoded = pd.get_dummies(df_binary, columns=categorical_cols, drop_first=False) # New column created per category as True or False
df_encoded = df_encoded.drop(columns=['Admin Status']) # Drop Admin Status due to data leakage
df_encoded.shape # Check new column count

(2837, 108)

In [12]:
# Run check on columns to confirm

[col for col in df_encoded.columns if 'Admin Status' in col or 'Student Status' in col or 'Department' in col]

['Department_Department A',
 'Department_Department B',
 'Department_Department C',
 'Student Status_Normal',
 'Student Status_Off-site',
 'Student Status_PR/Repeat']

In [13]:
# Final check on rows and columns

df_encoded.shape # Should remain as 2837, 110 (no rows lost or duplicated during the encoding)

(2837, 108)

In [14]:
# Dropping columns used for manual checks

df_encoded = df_encoded.drop(columns=['Notes', 'check'])
df_encoded.shape # check the two columns are dropped

(2837, 106)

In [15]:
# Final check on nulls (or greys)

null_check = df_encoded.isnull().sum() # Count any nulls per column
null_check[null_check > 0] # Show only the columns that have nulls

Attendance (%)                       52
Week 1 & 2 Attendance %             129
Week 2 & 3 Attendance %             106
Week 3 & 4 Attendance %              90
Week 4 & 5 Attendance %              86
Week 5 & 6 Attendance %              86
Week 6 & 7 Attendance %              84
Week 7 & 8 Attendance %              82
Week 8 & 9 Attendance %              81
Week 9 & 10 Attendance %             81
Week 11 & 12 Attendance %           110
Week 11 & 12 Attendance RAG          21
Week 11 & 12 VLE Engagement RAG      21
Week 11 & 12 COMBINED RAG            21
Week 12 & 13 Attendance %           110
Week 12 & 13 Attendance RAG          21
Week 12 & 13 VLE Engagement RAG      21
Week 12 & 13 COMBINED RAG            21
Week 13 & 14 Attendance %           110
Week 13 & 14 Attendance RAG          21
Week 13 & 14 VLE Engagement RAG      21
Week 13 & 14 COMBINED RAG            21
Week 14 & 15 Attendance %           110
Week 14 & 15 Attendance RAG          21
Week 14 & 15 VLE Engagement RAG      21


In [16]:
# X/y split

leakage_and_id_cols = ['Academic Year', 'Student_ID', 'Progression Decision', 'Resit?', 'Progression Decision (Resit)', 'is_repeating', 'repeating_with_attendance', 'repeating_with_no_attendance_expectation'] # Columns which will be excluded as information is not relevant to prediction
raw_rag_cols = [col for col in df_encoded.columns if 'RAG' in col and 'has_grey' not in col] # Gets all raw weekly RAG text columns as there is summary features
weekly_pct_cols = [col for col in df_encoded.columns if '&' in col and 'Attendance %' in col] # Gets all raw weekly attendance percentage columns as there is overall attendance percentage

cols_to_drop = leakage_and_id_cols + raw_rag_cols + weekly_pct_cols # Get all columns into one place to remove
X = df_encoded.drop(columns=cols_to_drop + ['initially_failed']) # Create new table but remove said columns as well as the true outcome (as that is the target)
y = df_encoded['initially_failed'] # Single target column

X.shape, y.shape # Print shapes

((2837, 21), (2837,))

In [17]:
# Check to ensure all features exist
X.columns.tolist()

['Attendance (%)',
 'is_offsite',
 'no_attendance_data',
 'unexplained_null_attendance',
 'vle_avg_score',
 'vle_red_weeks',
 'vle_grey_weeks',
 'has_grey_vle',
 'no_vle_data',
 'Department_Department A',
 'Department_Department B',
 'Department_Department C',
 'Student Status_Normal',
 'Student Status_Off-site',
 'Student Status_PR/Repeat',
 'Year Group_Fifth Year',
 'Year Group_First Year',
 'Year Group_Foundation Year',
 'Year Group_Fourth Year',
 'Year Group_Second Year',
 'Year Group_Third Year']

In [18]:
# Student Status counts of students with null

excluded_mask = X[['Attendance (%)', 'vle_avg_score']].isnull().any(axis=1) 

X_excluded_check = df_encoded[excluded_mask] # Pull all rows for the 103 students
X_excluded_check[['Student Status_Off-site', 'Student Status_PR/Repeat', 'Student Status_Normal']].sum() # Count how many students fall into each category

Student Status_Off-site     42
Student Status_PR/Repeat    60
Student Status_Normal        1
dtype: int64

In [19]:
# Implement null handing (exclude from data)

X_excluded = X[excluded_mask].copy() # Pull 103 students features into a seperate table
y_excluded = y[excluded_mask].copy() # Pull 103 outcomes for the students

X_modelling = X[~excluded_mask].copy() # Everyone else BUT the 103 students
y_modelling = y[~excluded_mask].copy() # Same as above for outcomes

X_modelling.shape, X_excluded.shape

((2734, 21), (103, 21))

In [20]:
# New Train & Test Split post exclusion - same steps as before just change of data

from sklearn.model_selection import train_test_split # Import the split function

X_train, X_test, y_train, y_test = train_test_split(
    X_modelling, y_modelling,
    test_size=0.2,
    stratify=y_modelling,
    random_state=22
)
X_train.shape, X_test.shape

((2187, 21), (547, 21))

In [21]:
# SMOTE application

from imblearn.over_sampling import SMOTE # Importing SMOTE for oversampling minority class

smote = SMOTE(random_state=22)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train) # Resamples the training data only leaving test untouched

y_train_smote.value_counts() 

initially_failed
False    1377
True     1377
Name: count, dtype: int64

In [23]:
# Cross-validation strategy and scorer set-up

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV # Import cross validation and search tools
from sklearn.metrics import make_scorer, recall_score # Import scoring tools

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=22) # 5-fold cross validation, keeps class balance per fold, same fixed seed
recall_scorer = make_scorer(recall_score) # Scoring tells the search to optimise for recall specifically

In [24]:
# Tuning Logistic Regression

from sklearn.linear_model import LogisticRegression # Import the model

logreg_params = {
    'C':[0.01, 0.1, 1, 10, 100], # C controls regularisation strength
    'penalty': ['l1', 'l2'], # Regularisation: try both to allow decision on which suits the data better
    'solver': ['liblinear'] # Underlying algorithm supporting both penalties above
}

logreg_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=22), # Base model: fixed max iterations and seed
    param_grid=logreg_params, # Combinations to try
    scoring=recall_scorer, # Optimise for recall (main aim of project)
    n_jobs=-1 # Use all available CPU cores to speed this up
)

logreg_search.fit(X_train_smote, y_train_smote) # Run the search on SMOTE-balanced training data

print('Best Parameters:', logreg_search.best_params_) # Prints the best combination
print('Best Cross-Validation Recall:', logreg_search.best_score_) # Print the recall score that combination achieved

Best Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Best Cross-Validation Recall: 0.6869644268774703


In [25]:
# Refit Logistic Regression with best parameters

logreg_tuned = logreg_search.best_estimator_ # Pulls the fitted model using best parameter combination

y_pred_logreg_tuned = logreg_tuned.predict(X_test) # Predict on the untouched test set

# Evaluate on untouched test set

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(classification_report(y_test, y_pred_logreg_tuned)) # Pecision, recall, f1-score for tuned model

y_fail_proba_logreg_tuned = logreg_tuned.predict_proba(X_test)[:,1] # Probabilities for AUC-ROC
print('AUC-ROC:', roc_auc_score(y_test, y_fail_proba_logreg_tuned))

              precision    recall  f1-score   support

       False       0.77      0.77      0.77       344
        True       0.61      0.62      0.61       203

    accuracy                           0.71       547
   macro avg       0.69      0.69      0.69       547
weighted avg       0.71      0.71      0.71       547

AUC-ROC: 0.7740147783251232


Tuning Logistic Regression with GridSearchCV (optimising for recall, as project aims) found the best parameters to be C = 0.1, penalty = l2 and solver = liblinear with a cross-validation recall of 0.687. However, when refit and evaluated on the actual test set, the recall only reached 0.62 (note that this is still a rise from 0.59 pre-tuning). This gap between the cross-validation estimate and the results is noteworthy - cross-validation recall is an average across 5 training folds and can be optimistic compared to how the actual model performs on data it has never seen. Overall, tuning did produce an improvement in recall with hardly any change in AUC-ROC (0.774 vs 0.773).

In [30]:
# Tuning Random Forest

from sklearn.ensemble import RandomForestClassifier # Import the model


rf_params = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_depth': [None, 10, 20, 30], # How deep each tree can grow
    'min_samples_split': [2, 5, 10], # Minimum number of samples required to split node
    'min_samples_leaf': [1, 2, 4], # Minimum number of samples allowed in a final leaf
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=22), # Base model, fixed seed
    param_distributions=rf_params, # Combinations to sample from
    n_iter=20, # Randomly try 20 combinations (as opposed to all possible)
    scoring=recall_scorer, # Optimise for recall
    cv=cv_strategy, # Same 5-fold CV
    random_state=22, # Makes the 20 combintions attempted reproducible
    n_jobs=-1 # Use all CPU cores
)

rf_search.fit(X_train_smote, y_train_smote) # Run the search

print('Best Parameters:', rf_search.best_params_) # Prints the best combination
print('Best Cross-Validation Recall:', rf_search.best_score_) # Print the recall score that combination achieved

Best Parameters: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 30}
Best Cross-Validation Recall: 0.7407088274044795


In [ ]:
# Refit Random Forest with best parameters

rf_tuned = rf_search.best_estimator_ # Pulls the fitted model using best parameter combination

y_pred_rf_tuned = rf_tuned.predict(X_test) # Predict on the untouched test set

# Evaluate on untouched test set

print(classification_report(y_test, y_pred_rf_tuned)) # Pecision, recall, f1-score for tuned model

y_fail_proba_rf_tuned = rf_tuned.predict_proba(X_test)[:,1] # Probabilities for AUC-ROC
print('AUC-ROC:', roc_auc_score(y_test, y_fail_proba_rf_tuned))

              precision    recall  f1-score   support

       False       0.77      0.78      0.78       344
        True       0.62      0.61      0.62       203

    accuracy                           0.72       547
   macro avg       0.70      0.69      0.70       547
weighted avg       0.72      0.72      0.72       547

AUC-ROC: 0.7701125558483217


Similarly to Logistic Regression, cross-validation recall promised 0.744 but the real results with the test data only moved marginally (from 0.60 to 0.61). This mirrors the same cross-validation versus test gap seen with Logistic Regression, suggesting this is a consistent pattern rather than a one-off for this dataset.

In [ ]:
# Tuining XGBoost

from xgboost import XGBClassifier # Import model

xgb_params = {
    'n_estimators': [100, 200, 300], # Number of boosting rounds
    'max_depth': [3, 5, 7, 9], # How deep each tree can grow
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # How much each new tree corrects the previous ones
    'subsample': [0.7, 0.8, 0.9, 1.0], # Fraction of training data randomly sampled for each tree
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=22, eval_metric='logloss'), # Base model, fixed seed, same setting as baseline
    param_distributions=xgb_params, # Combinations to sample from
    n_iter=20, # Randomly try 20 combinations
    scoring=recall_scorer, # Optimise for recall
    cv=cv_strategy, # Same 5-fold CV
    random_state=22, # Makes the 20 combintions attempted reproducible
    n_jobs=-1 # Use all CPU cores
)

xgb_search.fit(X_train_smote, y_train_smote) # Run the search

print('Best Parameters:', xgb_search.best_params_) # Prints the best combination
print('Best Cross-Validation Recall:', xgb_search.best_score_) # Print the recall score that combination achieved

Best Parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.2}
Best Cross-Validation Recall: 0.72834255599473


In [29]:
# Refit XGBoost with best parameters

xgb_tuned = xgb_search.best_estimator_ # Pulls the fitted model using best parameter combination

y_pred_xgb_tuned = xgb_tuned.predict(X_test) # Predict on the untouched test set

# Evaluate on untouched test set

print(classification_report(y_test, y_pred_xgb_tuned)) # Pecision, recall, f1-score for tuned model

y_fail_proba_xgb_tuned = xgb_tuned.predict_proba(X_test)[:,1] # Probabilities for AUC-ROC
print('AUC-ROC:', roc_auc_score(y_test, y_fail_proba_xgb_tuned))

              precision    recall  f1-score   support

       False       0.75      0.78      0.77       344
        True       0.60      0.57      0.59       203

    accuracy                           0.70       547
   macro avg       0.68      0.68      0.68       547
weighted avg       0.70      0.70      0.70       547

AUC-ROC: 0.7524630541871922


Post-tuning XGBoost's recall did not move significatly from the base model, and AUC-ROC actually decreased slightly, from 0.7654 to 0.7524. This may indicate that tuning did not meaningfully help XGBoost in this instance. It's also worth mentioning that the cross validation recall score was 0.7283, which once again dropped substantially against the test data to 0.57 - reinforcing the pattern seen with the other tuned models.

In [ ]:
# Tuning LightGBM

from lightgbm import LGBMClassifier # Import the model

lgbm_params = {
    'n_estimators': [100, 200, 300], # Number of boosting rounds
    'max_depth': [3, 5, 7, 9], # How deep each tree can grow
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # How much each new tree corrects the previous ones
    'num_leaves': [15, 31, 63], # Max number of leaves per tree
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(random_state=22), # Base model, fixed seed
    param_distributions=lgbm_params, # Combinations to sample from
    n_iter=20, # Randomly try 20 combinations
    scoring=recall_scorer, # Optimise for recall
    cv=cv_strategy, # Same 5-fold CV
    random_state=22, # Makes the 20 combintions attempted reproducible
    n_jobs=1 # Use all CPU cores
)

lgbm_search.fit(X_train_smote, y_train_smote) # Run the search

print('Best Parameters:', lgbm_search.best_params_) # Prints the best combination
print('Best Cross-Validation Recall:', lgbm_search.best_score_) # Print the recall score that combination achieved